## LLM based Agentic data flow

This Jupyter notebook implements a multi-agent system for handling telecom incident resolution and maintenance scheduling using Retrieval-Augmented Generation (RAG) and Large Language Models (LLMs).

## Key Components

1. **Imports and Setup**
   - Imports necessary libraries and modules
   - Sets up system path and loads constants

2. **Configuration**
   - Defines constants for Google Cloud Project, BigQuery, and Vertex AI

3. **Agent Definitions**
   - User Interface Agent: Handles initial user interactions
   - Incident Center Agent: Resolves user-reported incidents
   - Cell Tower Maintenance Agent: Schedules maintenance tasks

4. **Utility Functions**
   - `query_incident_resolution_database`: Searches for relevant incident resolution documents
   - `gen_detailed_resolution`: Generates detailed resolutions using Vertex AI
   - `submit_ticket`: Creates and stores incident tickets
   - `get_incident_ticket_info`: Retrieves ticket information
   - `classify_maintenance_task`: Classifies maintenance tasks using LLM
   - `schedule_maintenance_task`: Schedules maintenance based on task type

5. **Interactive Chat Interface**
   - Implements a widget-based chat interface for user interaction
   - Handles message processing and agent switching

## Workflow

1. User interacts with the User Interface Agent
2. Based on the query, the system transfers to either:
   - Incident Center Agent for resolving issues
   - Cell Tower Maintenance Agent for scheduling maintenance
3. Agents use RAG and LLMs to process queries and provide responses
4. The system maintains a chat history and context throughout the interaction

This notebook demonstrates an advanced use of LLMs and RAG for practical telecom incident management and maintenance scheduling tasks.

![agents_00](../../assets/agents_00.png)

In [1]:
#!pip install git+https://github.com/velascoluis/swarm-gemini.git

In [1]:
import sys
import json
import os
import uuid
from datetime import datetime, timedelta
from swarm import Swarm, Agent
import vertexai
from vertexai.generative_models import GenerativeModel
import pandas as pd
import numpy as np
import logging

In [2]:
sys.path.append(os.path.dirname(os.getcwd()))
from utils import run_query, load_constants

In [4]:
constants = load_constants()

GOOGLE_CLOUD_PROJECT = constants["GCP"]["GOOGLE_CLOUD_PROJECT"]
GOOGLE_CLOUD_LOCATION = constants["GCP"]["GOOGLE_CLOUD_LOCATION"]
GOOGLE_CLOUD_LOCATION_MULTI_REGION = constants["GCP"][
    "GOOGLE_CLOUD_LOCATION_MULTI_REGION"
]
GOOGLE_CLOUD_GCS_BUCKET = constants["GCP"]["GOOGLE_CLOUD_GCS_BUCKET"]
GOOGLE_CLOUD_GCS_BUCKET_MULTI_REGION = constants["GCP"][
    "GOOGLE_CLOUD_GCS_BUCKET_MULTI_REGION"
]
GOOGLE_GEMINI_MODEL_15 = constants["VERTEX"]["GOOGLE_GEMINI_MODEL_15"]
GOOGLE_GEMINI_MODEL_10 = constants["VERTEX"]["GOOGLE_GEMINI_MODEL_10"]

GOOGLE_CLOUD_BIGQUERY_PROJECT = constants["BIGQUERY"]["GOOGLE_CLOUD_BIGQUERY_PROJECT"]
GOOGLE_CLOUD_BIGQUERY_DATASET = constants["BIGQUERY"]["GOOGLE_CLOUD_BIGQUERY_DATASET"]
GOOGLE_CLOUD_BIGQUERY_DATASET_MULTI_REGION = constants["BIGQUERY"][
    "GOOGLE_CLOUD_BIGQUERY_DATASET_MULTI_REGION"
]


BASE_TABLE_NAME_EVENTS = constants["BIGQUERY"]["BASE_TABLE_NAME_EVENTS"]
BASE_TABLE_NAME_INCIDENTS = constants["BIGQUERY"]["BASE_TABLE_NAME_INCIDENTS"]


In [5]:
context_variables = {}

In [6]:
client = Swarm(
    llm_provider="vertexai",
    project_id=GOOGLE_CLOUD_PROJECT,
    location=GOOGLE_CLOUD_LOCATION,
)

In [8]:
INCIDENT_TICKETS_DATABASE_FILE = "incident_tickets.json"

def transfer_to_incident_center():
    """Transfer the user to the incident center agent to solve incidents."""
    return incident_center_agent


def transfer_to_cell_tower_maintenance():
    """Transfer the user to the cell tower maintenance agent to schedule maintenance tasks."""
    return cell_tower_maintenance_agent


def transfer_to_user_interface():
    """Transfer the user to the user interface agent to handle general questions or when no other agent is correct for the user query or when the work is done."""
    return user_interface_agent


def query_incident_resolution_database(user_incident):
    """Query the incidents resolution docs database for retrieving relevant docs to solve the user incident."""
    query_search = f"""
    SELECT *
    FROM VECTOR_SEARCH(
      TABLE `{GOOGLE_CLOUD_BIGQUERY_PROJECT}.{GOOGLE_CLOUD_BIGQUERY_DATASET_MULTI_REGION}.{BASE_TABLE_NAME_INCIDENTS}_docs_embedded`, 'ml_generate_embedding_result',
      (
      SELECT ml_generate_embedding_result, content AS query
      FROM ML.GENERATE_EMBEDDING(
      MODEL `{GOOGLE_CLOUD_BIGQUERY_DATASET_MULTI_REGION}.gecko_embedder`,
      (SELECT '{user_incident}' AS content))
      ),
  top_k => 5);"""
    results = run_query(query_search)
    return results["base"].to_string()


def gen_detailed_resolution(docs):
    """Generate a detailed resolution combining information from the docs provided."""
    vertexai.init(project=GOOGLE_CLOUD_PROJECT, location=GOOGLE_CLOUD_LOCATION)
    model = GenerativeModel(GOOGLE_GEMINI_MODEL_15)
    incident_resolution = model.generate_content(f"Generate a detailed resolution for the following incident: {docs}")
    return {"response": incident_resolution.text}


def submit_ticket(user_incident, incident_resolution):
    """Submit a ticket for the user."""
    ticket_id = str(uuid.uuid4())[:8] 
    ticket_entry = {
        "ticket_id": ticket_id,
        "user_incident": user_incident,
        "resolution": incident_resolution
    }

    try:
        with open(INCIDENT_TICKETS_DATABASE_FILE, "r+") as f:
            database = json.load(f)
            database.append(ticket_entry)
            f.seek(0)
            json.dump(database, f, indent=2)
            f.truncate()
    except FileNotFoundError:
        with open(INCIDENT_TICKETS_DATABASE_FILE, "w") as f:
            json.dump([ticket_entry], f, indent=2)

    return {
        "response": f"Ticket {ticket_id} created for '{user_incident}' with the provided resolution."
    }


def get_incident_ticket_info(ticket_id):
    """Get the information from an incident ticket."""
    database_file = "incident_tickets.json"
    try:
        with open(database_file, "r") as f:
            database = json.load(f)
        for ticket in database:
            if ticket["ticket_id"] == ticket_id:
                return ticket
        raise ValueError(f"Ticket with ID {ticket_id} not found")
    except FileNotFoundError:
        raise FileNotFoundError(f"Database file {database_file} not found")
    except json.JSONDecodeError:
        raise ValueError(f"Invalid JSON format in {database_file}")

def classify_maintenance_task(incident_ticket_info):
    """Classify the type of maintenance task based on the ticket information."""
    vertexai.init(project=GOOGLE_CLOUD_PROJECT, location=GOOGLE_CLOUD_LOCATION)
    model = GenerativeModel(GOOGLE_GEMINI_MODEL_15)
    VALID_MAINTENANCE_TASKS = ["tower_maintenance", "fiber_maintenance", "antenna_maintenance", "power_maintenance", "network_maintenance"]
    maintenance_task = model.generate_content(f"""Classify the type of maintenance task for the following incident: {incident_ticket_info}.
                                              Valid maintenance tasks are: {VALID_MAINTENANCE_TASKS}
                                              Always return one of the valid maintenance tasks.""")
    return {"response": maintenance_task.text}

def schedule_maintenance_task(maintenance_task):
    """Schedule the maintenance task."""
    next_available_date = _get_next_available_date(maintenance_task)
    return {"response": f"Maintenance task scheduled for {next_available_date}"}

def _get_next_available_date(maintenance_task):
    if maintenance_task == "tower_maintenance":
        return (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d")
    elif maintenance_task == "fiber_maintenance":
        return (datetime.now() + timedelta(days=2)).strftime("%Y-%m-%d")
    else:
        return (datetime.now() + timedelta(days=3)).strftime("%Y-%m-%d")


incident_center_agent = Agent(
    name="Incident Center Agent",
    instructions="""
    You are an Telco Root Case Analyst Agent, your goal is to assists users with incidents resolution.
     Always introduce yourself, greet the user and ask for the incident symptons.
    To resolve an incident:
     1. You need to search for the relevants docs where its explained how to solve the issue. Use as input the description of the incident and do not ask more information.
     2. With the docs retrieved, generate a detailed resolution for the incident.
     You can also open a new ticket with the details of the resolution.
     Do not ask for more detailed or specific information, just use the description of the incident provided by the user.
     """,
    functions=[
        query_incident_resolution_database,
        gen_detailed_resolution,
        submit_ticket,
        transfer_to_user_interface,
    ],
)

cell_tower_maintenance_agent = Agent(
    name="Cell Tower Maintenance Agent",
    instructions="""
    You are an Telco Cell Tower Maintenance Agent, your goal is to assist users to schedule maintenance tasks for cell towers.
    A visit is always required to perform the maintenance task.
    Always introduce yourself, greet the user and ask for the incident ticket id.
    Your instructions are:
    1. To schedule a new maintenance task you need first to retrieve information from a incident ticket id.
    2. Then you need to classify the type of maintenance task and the priority.
    3. Finally you need to schedule the maintenance task.
    """,
    functions=[
        get_incident_ticket_info,
        classify_maintenance_task,
        schedule_maintenance_task,
        transfer_to_user_interface,
    ],
)

user_interface_agent = Agent(
    name="User Interface Agent",
    instructions="""You are a user interface agent that handles all interactions with the user.
    Always introduce yourself greet the user and ask for their problem.
    Call this agent for general questions and when no other agent is correct for the user query.""",
    functions=[transfer_to_incident_center, transfer_to_cell_tower_maintenance],
)

In [9]:
"""
.The incident began with the core network operation center receiving alerts indicating a BGP peering session went down.
Concurrently, users reported experiencing service disruptions, including website inaccessibility and application connectivity issues
"""

'\n.The incident began with the core network operation center receiving alerts indicating a BGP peering session went down.\nConcurrently, users reported experiencing service disruptions, including website inaccessibility and application connectivity issues\n'

In [12]:
from ipywidgets import widgets
from IPython.display import display


input_box = widgets.Textarea(
    description="You:", placeholder="Type your message here...", continuous_update=False
)
input_box.layout.width = "80%"
input_box.style.description_width = "initial"
input_box.add_class("jupyter-widgets")
input_box.add_class("widget-text")
input_box.add_class("widget-text-copyable")
send_button = widgets.Button(description="Send")
output = widgets.Textarea(layout={"width": "90%", "height": "500px"})


def on_send(b):
    global chat_history, client, agent, messages, context_variables

    user_input = input_box.value
    input_box.value = ""

    if user_input.lower() in ["exit", "quit", "bye"]:
        chat_history += "AI: Goodbye!\n"
        output.value = chat_history
        send_button.disabled = True
        input_box.disabled = True
        return

    chat_history += f"You: {user_input}\n"
    messages.append({"role": "user", "content": user_input})

    response = client.run(
        agent=agent,
        messages=messages,
        context_variables=context_variables or {},
        stream=False,
        debug=True,
    )

    for message in response.messages:
        if message["role"] == "assistant":
            ai_response = f"AI: {message['content']}"
            chat_history += ai_response + "\n"

    output.value = chat_history
    messages.extend(response.messages)
    agent = response.agent


send_button.on_click(on_send)
display(widgets.VBox([output, widgets.HBox([input_box, send_button])]))

client = Swarm(
    llm_provider="vertexai",
    project_id=GOOGLE_CLOUD_PROJECT,
    location=GOOGLE_CLOUD_LOCATION,
)
agent = user_interface_agent
messages = []
chat_history = ""

[2024-10-24 10:33:46] Getting chat completion for...: [{'role': 'system', 'content': 'You are a user interface agent that handles all interactions with the user.\n    Always introduce yourself greet the user and ask for their problem.\n    Call this agent for general questions and when no other agent is correct for the user query.'}, {'role': 'user', 'content': 'Hi'}]
[2024-10-24 10:33:48] Received completion: ChatCompletionMessage(content="Hello! 👋 I'm your friendly user interface agent. What can I help you with today? 😊 \n", refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None)
[2024-10-24 10:33:48] Ending turn.
[2024-10-24 10:34:00] Getting chat completion for...: [{'role': 'system', 'content': 'You are a user interface agent that handles all interactions with the user.\n    Always introduce yourself greet the user and ask for their problem.\n    Call this agent for general questions and when no other agent is correct for the user query.'}, {'role': 'user'